### Feature Engineering

This notebook prepares the household shopping features for clustering by selecting distinct behaviors, using logged versions of heavily skewed basket measures, and standardizing the final inputs. Total spending is kept only for later cluster profiling, while the final model features represent shopping frequency, recency, cadence, basket value, and basket size.

#### Import libraries
The libraries needed to prepare the clustering features.

In [1]:
# Core analysis libraries
import pandas as pd
import numpy as np

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Improve plot styling
sns.set()

# Show all columns during EDA
pd.set_option('display.max_columns', None)

# Display all dataframes generated within a cell
from IPython.display import display

In [2]:
# File path locations
from pathlib import Path

# File paths
PROJECT_ROOT = Path.cwd().resolve().parent
CLEAN_DATA_DIR = PROJECT_ROOT / "data/clean_data"
RAW_DATA_DIR = PROJECT_ROOT / "data/raw_data"
MODELS_DIR = PROJECT_ROOT / "models"
SRC_DIR = PROJECT_ROOT / "src"
DEPLOYMENT_DIR = PROJECT_ROOT / "deployment"

#### Import data
The household-level dataset from the analysis notebook is loaded and checked for its shape, data types, and missing values.

In [3]:
# Import data and show first five rows
hh_shopping = pd.read_parquet(CLEAN_DATA_DIR / "02_household_shopping_data_analysis.parquet")
hh_shopping.head()

,household_key,num_trips,recency_days,total_spend,avg_basket_value,total_units_purchased,avg_days_between_trips,avg_basket_size,log10_total_spend,log10_avg_basket_value,log10_avg_basket_size
0,1,86,5,4330.16,50.35,1997,7.7,23.2,3.636504,1.701999,1.365488
1,2,45,43,1954.34,43.43,834,12.8,18.5,3.291000,1.637790,1.267172
2,3,47,8,2653.21,56.45,8540,12.8,181.7,3.423772,1.751664,2.259355
3,4,30,84,1200.11,40.00,382,18.0,12.7,3.079221,1.602060,1.103804
4,5,40,8,779.06,19.48,245,15.8,6.1,2.891571,1.289589,0.785330


In [4]:
# Number of rows, columns
hh_shopping.shape

(2500, 11)

In [5]:
# Data types and nulls
hh_shopping_types_null_counts = pd.DataFrame()
hh_shopping_types_null_counts["Data Type"] = hh_shopping.dtypes
hh_shopping_types_null_counts["Count of Nulls"] = hh_shopping.isna().sum()
hh_shopping_types_null_counts

,Data Type,Count of Nulls
household_key,int64,0
num_trips,int64,0
recency_days,int64,0
total_spend,float64,0
avg_basket_value,float64,0
total_units_purchased,int64,0
avg_days_between_trips,float64,0
avg_basket_size,float64,0
log10_total_spend,float64,0
log10_avg_basket_value,float64,0


#### Feature selection
The selected features capture how often households shop, how recently they shopped, their usual shopping cadence, and their typical basket value and size. Total spend and total units are left out of the clustering inputs because they overlap with the frequency and average-based features.

In [6]:
model_feature_names = [
    "num_trips", 
    "recency_days", 
    "avg_days_between_trips", 
    "log10_avg_basket_size",
    "log10_avg_basket_value"
]

model_features = hh_shopping[model_feature_names].copy()

#### Transform features
The logged versions of average basket value and average basket size are used because their original distributions were heavily right-skewed. All five clustering features are then standardized so their different numerical scales do not affect the clustering results.

In [7]:
# Import transformer libraries
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

In [8]:
# Create preprocessor object to automate tranformations
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), model_feature_names)
    ],
    verbose_feature_names_out=False
)

In [9]:
# Fit and transform model features
model_features_processed = preprocessor.fit_transform(model_features)
model_features_processed

array([[-0.21266236, -0.32772577, -0.23887154, -0.72866064,  1.07008879],
       [-0.56719185,  0.27758089, -0.09102127, -0.82931923,  0.81967741],
       [-0.54989773, -0.2799384 , -0.09102127,  0.18650214,  1.26377541],
       ...,
       [ 0.53098488, -0.39144226, -0.36063058,  0.18772401, -0.96628666],
       [-0.17807411, -0.37551314, -0.25336666,  1.15207721,  0.5804835 ],
       [ 0.02080829, -0.35958402, -0.29975106,  1.29316485,  1.00843039]])

In [10]:
# Check number of rows and columns to confirm no data lost
model_features_processed.shape

(2500, 5)

In [11]:
# Create dataframe of transformed features
feature_names = preprocessor.get_feature_names_out()

model_features_processed_df = pd.DataFrame(
    model_features_processed, 
    columns=feature_names, 
    index=model_features.index
)

#### Add identification and profiling fields
Household key and logged total spending are added back after scaling so households can be identified and the clusters can be profiled later. Neither field will be used as an input to the clustering models.

In [12]:
# Add 'household_key' and 'log10_total_spend' back to dataframe
model_features_processed_df["household_key"] = hh_shopping["household_key"]
model_features_processed_df["log10_total_spend"] = hh_shopping["log10_total_spend"]

# Re-order so 'household_key' is the first column listed
model_features_processed_df.insert(
    0, 
    "household_key", 
    model_features_processed_df.pop("household_key")
)

In [13]:
model_features_processed_df.head()

,household_key,num_trips,recency_days,avg_days_between_trips,log10_avg_basket_size,log10_avg_basket_value,log10_total_spend
0,1,-0.212662,-0.327726,-0.238872,-0.728661,1.070089,3.636504
1,2,-0.567192,0.277581,-0.091021,-0.829319,0.819677,3.291000
2,3,-0.549898,-0.279938,-0.091021,0.186502,1.263775,3.423772
3,4,-0.696898,0.930675,0.059728,-0.996579,0.680335,3.079221
4,5,-0.610427,-0.279938,-0.004051,-1.322641,-0.538272,2.891571


#### Export data
The completed dataset is exported with the five scaled clustering features and the separate fields needed for household identification and profiling.

In [14]:
model_features_processed_df.to_parquet(CLEAN_DATA_DIR / "03_model_features_processed.parquet")

#### Save preprocessor
The fitted preprocessor is saved so the same feature order and scaling can be reused in the modeling notebook.

In [15]:
import joblib

joblib.dump(preprocessor, MODELS_DIR / "dunnhumby_preprocessor.joblib")

['C:\\Users\\jac67\\Documents\\Data and Analytics\\Python\\dunnhumby-segmentation\\models\\dunnhumby_preprocessor.joblib']